# P1.1 rain-only reference population and P1.2 rain-only expectation model (Ogallala)

Driver for the `paper1` package (research plan revision 2, section 5.3, updated to plan v3 cards P1.0-P1.2). The code lives in the folder `paper1`; this notebook only sets the run up, calls it, and shows what it produced.

**Before the first run**
1. Copy the repository folder `paper1` to Google Drive as `MyDrive/MSUGWB/code/paper1` (the whole folder; re-copy after every code change - outputs record a hash of the code they were made with).
2. Inputs must be on Drive where `paper1/config.py` (`DATASETS`) expects them (checked 2026-11-11): Landsat `LANDSAT_C2L2_indices_30m_monthly_tiles_v2_latest` (2015-2026 so far), OpenET `OPENET_v2_0_monthly_30m_tiles`, CDL `USDA_NASS_CDL_30m_tiles` (each with `frames.json`), the NWI tiles (made by `Colab notebooks/NWI_Wetlands_30m_Ogallala.ipynb`), `GRIDMET_LAB_NEW/GRIDMET_Ogallala_lab.nc` and `GRIDMET_DROUGHT_Ogallala_lab.nc`, the GLO-30 DEM in `MyDrive/MGU GWB datasets/merged_datasets (1)/`, `HRES_WTD/HRES_WTD_Ogallala.nc`, `LANID/...nc`, gSSURGO 120 m and GDE in `USGS data/derived_usgs/`, and `MyDrive/high_plains_quifer.zip`. NHDPlus HR, the Copernicus DEM tiles for HAND and the Census counties are downloaded by the pipeline.
3. Use a GPU runtime for P1.2 (Runtime > Change runtime type). P1.1 runs on CPU. P1.2 needs `calendars.season_window` written first.

**Order:** `RUN_MODE = "test"` first (minutes; a 30 km window of one tile, 2015-2016), read the reports, then `"focus"` (focus-landscape tiles) or `"full"`. In `focus`/`full` mode a failed gate (G1a, G1b) halts the stages after it unless listed in `GATE_OVERRIDE`.

**Monitoring:** cell 9 prints the stage table at any time; the log is `<outputs>/logs/run_<date>.log`, one line per unit of work in `<outputs>/ledger.jsonl`. A stage that hits the session time budget stops cleanly - rerun the same cell in a new session and it continues. See `paper1/docs/MONITORING.md`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ═══════════════ CELL 1: settings (edit these, then Run all) ═══════════════
RUN_MODE = "test"            # "test" | "focus" | "full"
WHAT = ["p11"]               # "p11" (Step 0 + P1.1), "p12" (P1.2), "all", or stage names e.g. ["tiles", "blocks"]
FORCE = False                # True recomputes the named stages even if complete (rarely needed: settings changes get new folders)
CODE_ON_DRIVE = "/content/drive/MyDrive/MSUGWB/code/paper1"
STOP_ON_PREFLIGHT_PROBLEMS = True
RUN_TESTS = True             # unit tests before, provenance tests after the run

In [4]:
# ═══════════════ CELL 2: install ═══════════════
!pip install -q "pysheds==0.5" netCDF4 rasterio pyogrio 2>&1 | tail -2

In [5]:
# ═══════════════ CELL 3: mount Drive, load the code ═══════════════
import os, sys, shutil, hashlib, json
if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
assert os.path.isdir(CODE_ON_DRIVE), f"copy the paper1 folder to {CODE_ON_DRIVE} first"
LOCAL_CODE = "/content/code/paper1"
shutil.rmtree(LOCAL_CODE, ignore_errors=True)
shutil.copytree(CODE_ON_DRIVE, LOCAL_CODE, ignore=shutil.ignore_patterns("__pycache__", "*.pyc"))
os.environ["P1_RUN_ENV"], os.environ["P1_RUN_MODE"] = "colab", RUN_MODE
if "/content/code" not in sys.path:
    sys.path.insert(0, "/content/code")
for m in [m for m in sys.modules if m == "paper1" or m.startswith("paper1.")]:
    del sys.modules[m]                       # pick up a re-copied folder without restarting the runtime
import paper1.config as C
C.RUN_ENV, C.RUN_MODE = "colab", RUN_MODE
h = hashlib.md5()
for root, _, files in sorted(os.walk(LOCAL_CODE)):
    for f in sorted(files):
        if f.endswith(".py"):
            h.update(open(os.path.join(root, f), "rb").read())
print("paper1 code md5:", h.hexdigest()[:12], "| outputs:", C.out_root())

paper1 code md5: e0e924e67dc9 | outputs: /content/drive/MyDrive/MSUGWB/P1_rain_only/_test


In [6]:
# ═══════════════ CELL 4: preflight (inputs, sizes, libraries, GPU, Drive read speed) ═══════════════
from paper1.__main__ import preflight
rep = preflight()
print(json.dumps({k: rep.get(k) for k in ("run_mode", "tiles", "tile_files", "read_mb_per_s", "scratch_free_gb", "gpu", "libraries")}, indent=1, default=str))
for n in rep.get("notes", []):
    print("NOTE:", n)
for p in rep["problems"]:
    print("PROBLEM:", p)
if rep["problems"] and STOP_ON_PREFLIGHT_PROBLEMS:
    raise SystemExit("preflight found problems (listed above); fix them or set STOP_ON_PREFLIGHT_PROBLEMS = False")

{
 "run_mode": "test",
 "tiles": [
  "z14e02n29"
 ],
 "tile_files": {
  "present": 6,
  "missing": [
   "/content/drive/MyDrive/MSUGWB/NWI_wetlands_30m_tiles/z14e02n29/NWI_wetlands_30m_tiles_z14e02n29_Ogallala.nc"
  ],
  "gb": 19.8
 },
 "read_mb_per_s": 52.8,
 "scratch_free_gb": 241.0,
 "gpu": null,
 "libraries": {
  "numpy": "2.4.6",
  "netCDF4": "1.7.5.1.post1",
  "pyproj": "3.8.0",
  "rasterio": "1.5.2",
  "geopandas": "1.2.0",
  "pyogrio": "0.13.0",
  "shapely": "2.2.0",
  "scipy": "1.16.3",
  "pysheds": "0.5",
  "requests": "2.32.4",
  "matplotlib": "3.10.0",
  "torch": "2.14.1+cpu"
 }
}
PROBLEM: gridmet: processing_status is not 'complete'
PROBLEM: 1 tile files missing (see tile_files.missing)


SystemExit: preflight found problems (listed above); fix them or set STOP_ON_PREFLIGHT_PROBLEMS = False

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ═══════════════ CELL 5: unit tests (seconds) ═══════════════
if RUN_TESTS:
    !cd /content/code && P1_RUN_ENV=colab P1_RUN_MODE={RUN_MODE} python -m paper1 test unit

In [ ]:
# ═══════════════ CELL 6: run ═══════════════
from paper1.pipeline import runner, P11, P12
from paper1.__main__ import print_status
expand = dict(p11=P11, p12=P12, all=P11 + P12)
targets = [s for n in WHAT for s in expand.get(n, [n])]
try:
    runner().run(targets, force=FORCE)
finally:
    print_status()

In [ ]:
# ═══════════════ CELL 7: provenance tests on what was just produced ═══════════════
if RUN_TESTS:
    !cd /content/code && P1_RUN_ENV=colab P1_RUN_MODE={RUN_MODE} python -m paper1 test provenance

In [ ]:
# ═══════════════ CELL 8: reports ═══════════════
from IPython.display import Markdown, Image, display
from paper1.tests.util import stage_dir
for st, name in (("verify_p11", "verify_p11.md"), ("verify_p12", "verify_p12.md")):
    d = stage_dir(st)
    if d:
        display(Markdown(open(f"{d}/reports/{name}").read()))
        pngs = sorted(f for f in os.listdir(f"{d}/reports") if f.endswith(".png"))
        if pngs:
            display(Image(f"{d}/reports/{pngs[-1]}", width=520))
    else:
        print(f"{st}: not complete for RUN_MODE={RUN_MODE}")

In [ ]:
# ═══════════════ CELL 9: status (run any time) ═══════════════
from paper1.__main__ import print_status
print_status()
!tail -n 15 "{C.out_root()}/logs/"*.log

In [ ]:
# ═══════════════ CELL 10: reading the outputs (example) ═══════════════
# import xarray as xr
# ref = xr.open_dataset(f"{stage_dir('reference')}/z14e02n29/P1_reference_z14e02n29.nc")       # P1.1, (year, by, bx)
# exp = xr.open_dataset(f"{stage_dir('predict')}/2015/z14e02n29/P1_expectation_z14e02n29_2015.nc")  # P1.2, (month, by, bx)
# exp["ET_q50"].isel(month=6).plot()